# Convergence v1: the GRU baseline and the step 4a method trained on all the data

Trains `gru_uniform_full` (baseline) and `gru_anytime_uniform_full` (confidence head, consistency loss, two prefixes)
on every training drawing for about 3.75 times the earlier number of steps, two seeds each
(`configs/convergence_v1.yaml`). Then scores them on when to guess, together with the 600,000-drawing runs from
their saved predictions. **Validation data only.** Every comparison was fixed beforehand.

**Before running:**
1. In **Input**: your Kaggle Dataset with the processed data, and `ai-pictionary-step4a-output` **pinned to
   version 4** (holds `runs/predictions_all/*.npz`). The notebook stops early if it cannot find them.
2. **Internet on**. **GPU T4 x2**, then **Save Version → Save & Run All**. Expected: about 3-5 hours.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping before using GPU quota")

import numpy as np

# Saved VALIDATION predictions of the 600,000-drawing runs: a folder named predictions_all.
OLD_PREDICTIONS = None
for p in Path("/kaggle/input").rglob("gru_uniform_s42.npz"):
    if p.parent.name == "predictions_all":
        OLD_PREDICTIONS = p.parent
        break
needed = [f"{c}_s{s}.npz" for c in ("gru_uniform", "gru_anytime_uniform") for s in (42, 43)]
if OLD_PREDICTIONS is None or not all((OLD_PREDICTIONS / n).is_file() for n in needed):
    raise RuntimeError("runs/predictions_all with the gru_uniform and gru_anytime_uniform runs not found: "
                       "attach ai-pictionary-step4a-output pinned to version 4")
for n in needed:
    halves = set(np.unique(np.load(OLD_PREDICTIONS / n)["half"]).tolist())
    if halves != {0, 1}:
        raise RuntimeError(f"{n} is not a validation prediction file (halves {halves})")
print("saved validation predictions:", OLD_PREDICTIONS)


In [ ]:
MAX_MINUTES = 600
TOP = Path("/kaggle/working/runs/convergence_v1")
!PYTHONPATH=src python -m quickdraw_exp.experiments --matrix configs/convergence_v1.yaml --max-minutes {MAX_MINUTES}
# Checkpoint predictions of the new models on the validation halves (GPU).
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/convergence_v1.yaml --runs {TOP} --out {TOP}/predictions --num-workers 4


In [ ]:
ALL = Path("/kaggle/working/runs/predictions_convergence")
ALL.mkdir(parents=True, exist_ok=True)
for p in [*(OLD_PREDICTIONS / n for n in needed), *(TOP / "predictions").glob("*.npz")]:
    shutil.copy(p, ALL / p.name)
print(sorted(p.name for p in ALL.glob("*.npz")))
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/convergence_v1.yaml --predictions {ALL} --out /kaggle/working/runs/convergence_v1_anytime


In [ ]:
import json

training = json.loads((TOP / "results.json").read_text())
print("training", "COMPLETE" if training["complete"] else "INCOMPLETE, missing: " + ", ".join(training["missing"]))
old = {**json.loads(Path("reports/followup_v1/results.json").read_text())["table"],
       **json.loads(Path("reports/followup_v2/results.json").read_text())["table"]}
print("\nTop-1 by ink fraction, val 1000 per class (600k runs from reports/):")
for name, row in [("gru_uniform", old["gru_uniform"]), ("gru_anytime_uniform", old["gru_anytime_uniform"]),
                  *training["table"].items()]:
    print(f"  {name:26s} seeds {row['seeds']}", {f: round(v["top1"]["mean"], 4) for f, v in row["by_frac"].items()},
          "early", round(row["early_top1"]["mean"], 4))
for run in sorted(TOP.glob("*_s*/summary.json")):
    s = json.loads(run.read_text())
    print(f"  {run.parent.name}: steps {s.get('steps')}, epochs {s.get('epochs_done')}, "
          f"{s.get('train_samples_per_s')} samples/s")
print("\nValidation mean top-1 after each epoch (checkpoint selection):")
for log in sorted(TOP.glob("*_s*/events.log")):
    lines = [l for l in log.read_text().splitlines() if "val mean top-1" in l]
    if lines:
        print(f"  {log.parent.name}: " + " | ".join(l.split("epoch", 1)[-1].strip() for l in lines))

result = json.loads(Path("/kaggle/working/runs/convergence_v1_anytime/results.json").read_text())
print("\nmissing conditions:", result["missing_conditions"])
print("\nPre-registered comparisons (AUC, evaluation half):")
for row in result["comparisons"]:
    if row["difference"] is None:
        print(f"  {row['name']}: not available")
    else:
        print(f"  {row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, difference {row['difference']:+.4f}  "
              f"(a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")
print("\nRisk control and AUC per condition:")
for condition, policies in result["table"].items():
    for policy, e in policies.items():
        if "earliness_at_gap_0.02" in e:
            print(f"  {condition:26s} {policy:7s} seeds {e['seeds']} auc {e['auc']['mean']:.4f}  earliness@gap5% "
                  f"{e['earliness_at_gap_0.05']['mean']:.3f} met {e['met_gap_0.05']}")
for run, ref in result["reference"].items():
    print(f"  {run}: end of drawing {ref['end_of_drawing']['accuracy']:.4f}")
for name in training["missing"]:
    log = TOP / f"{name}.log"
    if log.exists():
        print(f"--- {name} ---")
        print("\n".join(log.read_text().splitlines()[-15:]))
